# EconoCausal: Dynamic Pricing Using Double Machine Learning

## Project Objective
To investigate how product price affects the number of units sold using Double Machine Learning (DML).

## Research Question
How does product price influence product sales?

## Methodology
- Dataset preparation and validation
- Treatment and outcome selection
- Confounder assessment
- Double Machine Learning estimation
- Effect interpretation and limitations

## Variables
- Treatment: Price
- Outcome: Units Sold
- Potential controls: To be finalized after checking variable timing and causal relevance.

**Important:** Predictive relationships do not automatically establish causal effects.

In [9]:
# Import required libraries

import pandas as pd
import numpy as np
from pathlib import Path

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

print("Libraries imported successfully!")
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

Libraries imported successfully!
Pandas version: 3.0.6
NumPy version: 2.5.4


## 1. Load the Prepared Dataset

In this section, we load the cleaned dataset and inspect its structure, columns, and sample records before selecting causal variables.

In [10]:
# Locate the processed dataset

possible_paths = [
    Path("data/processed/causal_data.csv"),
    Path("../../data/processed/causal_data.csv")
]

DATA_PATH = next(
    (path.resolve() for path in possible_paths if path.exists()),
    None
)

if DATA_PATH is None:
    raise FileNotFoundError(
        "causal_data.csv was not found. Check the data/processed folder."
    )

# Load dataset
df = pd.read_csv(DATA_PATH, parse_dates=["Date"])

print("Dataset loaded successfully!")
print("File location:", DATA_PATH)
print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst five records:")
display(df.head())

Dataset loaded successfully!
File location: C:\Users\methu\Desktop\EconoCausal\data\processed\causal_data.csv
Dataset shape: (76000, 18)

Column names:
['Date', 'Store ID', 'Product ID', 'Category', 'Region', 'Inventory Level', 'Units Sold', 'Units Ordered', 'Price', 'Discount', 'Weather Condition', 'Promotion', 'Competitor Pricing', 'Seasonality', 'Epidemic', 'Demand', 'Month', 'DayOfWeek']

First five records:


,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Price,Discount,Weather Condition,Promotion,Competitor Pricing,Seasonality,Epidemic,Demand,Month,DayOfWeek
0,2022-01-01,S001,P0001,Electronics,North,195,102,252,72.72,5,Snowy,0,85.73,Winter,0,115,1,5
1,2022-01-01,S001,P0002,Clothing,North,117,117,249,80.16,15,Snowy,1,92.02,Winter,0,229,1,5
2,2022-01-01,S001,P0003,Clothing,North,247,114,612,62.94,10,Snowy,1,60.08,Winter,0,157,1,5
3,2022-01-01,S001,P0004,Electronics,North,139,45,102,87.63,10,Snowy,0,85.19,Winter,0,52,1,5
4,2022-01-01,S001,P0005,Groceries,North,152,65,271,54.41,0,Snowy,0,51.63,Winter,0,59,1,5


In [11]:
# Dataset structure
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

print("\nComplete column list:")
for number, column in enumerate(df.columns, start=1):
    print(f"{number}. {column}")

print("\nData types:")
display(df.dtypes.to_frame(name="Data Type"))

print("\nMissing values:")
display(df.isnull().sum().to_frame(name="Missing Values"))

Number of rows: 76000
Number of columns: 18

Complete column list:
1. Date
2. Store ID
3. Product ID
4. Category
5. Region
6. Inventory Level
7. Units Sold
8. Units Ordered
9. Price
10. Discount
11. Weather Condition
12. Promotion
13. Competitor Pricing
14. Seasonality
15. Epidemic
16. Demand
17. Month
18. DayOfWeek

Data types:


,Data Type
Date,datetime64[us]
Store ID,str
Product ID,str
Category,str
Region,str
Inventory Level,int64
Units Sold,int64
Units Ordered,int64
Price,float64
Discount,int64



Missing values:


,Missing Values
Date,0
Store ID,0
Product ID,0
Category,0
Region,0
Inventory Level,0
Units Sold,0
Units Ordered,0
Price,0
Discount,0


In [12]:
print("ALL 18 COLUMNS:\n")

for i, col in enumerate(df.columns, start=1):
    print(f"{i}. {col}")

print("\nMISSING VALUES:")
print(df.isnull().sum().to_string())

ALL 18 COLUMNS:

1. Date
2. Store ID
3. Product ID
4. Category
5. Region
6. Inventory Level
7. Units Sold
8. Units Ordered
9. Price
10. Discount
11. Weather Condition
12. Promotion
13. Competitor Pricing
14. Seasonality
15. Epidemic
16. Demand
17. Month
18. DayOfWeek

MISSING VALUES:
Date                  0
Store ID              0
Product ID            0
Category              0
Region                0
Inventory Level       0
Units Sold            0
Units Ordered         0
Price                 0
Discount              0
Weather Condition     0
Promotion             0
Competitor Pricing    0
Seasonality           0
Epidemic              0
Demand                0
Month                 0
DayOfWeek             0


In [13]:
print("Total missing values:", df.isnull().sum().sum())
print("Treatment column exists:", "Price" in df.columns)
print("Outcome column exists:", "Units Sold" in df.columns)

Total missing values: 0
Treatment column exists: True
Outcome column exists: True


In [14]:
# Check price variation across products and stores

price_variation = (
    df.groupby(["Store ID", "Product ID"])["Price"]
    .agg(["count", "nunique", "min", "max", "std"])
)

print("Total store-product combinations:", len(price_variation))

print(
    "Combinations with changing prices:",
    (price_variation["nunique"] > 1).sum()
)

print(
    "Combinations with fixed prices:",
    (price_variation["nunique"] == 1).sum()
)

print("\nSample price variation:")
display(price_variation.head(10))

Total store-product combinations: 100
Combinations with changing prices: 100
Combinations with fixed prices: 0

Sample price variation:


count  nunique    min     max        std
Store ID Product ID                                          
S001     P0001         760      681  43.36   90.74   8.832447
         P0002         760      684  37.73   99.74   9.891345
         P0003         760      649  35.11   81.12   7.616094
         P0004         760      709  64.91  142.17  13.987488
         P0005         760      690  36.98   90.20   8.851224
         P0006         760      643  22.47   57.41   6.037307
         P0007         760      592  14.92   37.09   3.774378
         P0008         760      640  24.47   64.22   5.716501
         P0009         760      689  49.07  167.34  12.802284
         P0010         760      707  80.83  177.46  15.168571

In [15]:
print("Total store-product combinations:", len(price_variation))

print(
    "Combinations with changing prices:",
    (price_variation["nunique"] > 1).sum()
)

print(
    "Combinations with fixed prices:",
    (price_variation["nunique"] == 1).sum()
)

print(
    "Average unique prices per combination:",
    round(price_variation["nunique"].mean(), 2)
)

Total store-product combinations: 100
Combinations with changing prices: 100
Combinations with fixed prices: 0
Average unique prices per combination: 653.86


In [16]:
# Examine the treatment variable: Price

print("Price distribution:")
display(df["Price"].describe().round(2))

print("\nTotal unique prices:", df["Price"].nunique())

print("\nPrice quantiles:")
display(
    df["Price"].quantile(
        [0, 0.10, 0.25, 0.50, 0.75, 0.90, 1]
    ).round(2)
)

Price distribution:


count    76000.00
mean        67.73
std         39.38
min          4.74
25%         32.00
50%         64.50
75%         95.83
max        228.03
Name: Price, dtype: float64


Total unique prices: 15396

Price quantiles:


0.00      4.74
0.10     19.86
0.25     32.00
0.50     64.50
0.75     95.83
0.90    121.92
1.00    228.03
Name: Price, dtype: float64

## 2. Dataset Source and Limitations

### Dataset Source
- Platform: Kaggle
- Dataset: Retail sales and inventory data
- File used: sales_data.csv
- Records: 76,000
- Features: 18

### Research Variables
- Treatment: Price
- Outcome: Units Sold

### Data Limitations
- Dataset provenance and exact original file version require confirmation.
- The dataset may be synthetic.
- Observational relationships do not establish causation.
- Potential confounders must be evaluated before DML estimation.
- Variable timing must be verified to avoid post-treatment bias.

## 3. Investigating Potential Confounders

We examine whether discounts, promotions, and competitor pricing are associated with the product price and sales outcome.

These relationships are descriptive and do not establish that the variables are valid causal controls.

In [17]:
# Compare price and sales across promotion status

promotion_analysis = (
    df.groupby("Promotion")
    .agg(
        Average_Price=("Price", "mean"),
        Average_Units_Sold=("Units Sold", "mean"),
        Record_Count=("Price", "count")
    )
    .round(2)
)

display(promotion_analysis)

# Compare price and sales across discount levels

discount_analysis = (
    df.groupby("Discount")
    .agg(
        Average_Price=("Price", "mean"),
        Average_Units_Sold=("Units Sold", "mean"),
        Record_Count=("Price", "count")
    )
    .round(2)
)

display(discount_analysis)

,Average_Price,Average_Units_Sold,Record_Count
Promotion,,,
0,68.92,81.83,51000
1,65.30,103.10,25000


,Average_Price,Average_Units_Sold,Record_Count
Discount,,,
0,73.07,81.57,17126
5,68.53,81.69,16918
10,66.93,87.66,23298
15,67.08,102.88,6245
20,63.45,103.77,6191
25,58.74,103.59,6222


In [18]:
# Relationship between our price and competitor pricing

print("Price vs Competitor Pricing correlation:")

correlation = df[
    ["Price", "Competitor Pricing"]
].corr()

display(correlation.round(3))

# Summary statistics
display(
    df[["Price", "Competitor Pricing"]]
    .describe()
    .round(2)
)

Price vs Competitor Pricing correlation:


,Price,Competitor Pricing
Price,1.000,0.977
Competitor Pricing,0.977,1.000


,Price,Competitor Pricing
count,76000.00,76000.00
mean,67.73,69.45
std,39.38,40.94
min,4.74,4.29
25%,32.00,32.62
50%,64.50,65.70
75%,95.83,97.93
max,228.03,261.22


In [19]:
# Analyze promotion and competitor pricing

promotion_competitor = (
    df.groupby("Promotion")
    .agg(
        Average_Price=("Price", "mean"),
        Average_Competitor_Price=("Competitor Pricing", "mean"),
        Average_Units_Sold=("Units Sold", "mean"),
        Records=("Price", "count")
    )
    .round(2)
)

display(promotion_competitor)

,Average_Price,Average_Competitor_Price,Average_Units_Sold,Records
Promotion,,,,
0,68.92,70.64,81.83,51000
1,65.30,67.03,103.10,25000


## 4. Preliminary Findings: Pricing and Promotions

### Discount Analysis
- Higher discount levels were generally associated with higher average sales.
- Average selling prices tended to decrease as discounts increased.

### Promotion Analysis
- Records without promotion: 51,000
- Records with promotion: 25,000
- Average sales without promotion: 81.83 units
- Average sales with promotion: 103.10 units
- Average price without promotion: 68.92
- Average price with promotion: 65.30

### Competitor Pricing
- Correlation between Price and Competitor Pricing: 0.977.
- This indicates a very strong positive linear association.

### Important Limitation
These findings are descriptive. The timing and causal role of potential control variables must be established before DML estimation.

## 5. Preliminary DML Variable Strategy

### Treatment
Price

### Outcome
Units Sold

### Potential Control Variables
Store ID, Product ID, Date, Promotion, Competitor Pricing,
Weather Condition, Seasonality, and Region.

### Variables Requiring Further Investigation
Discount, Category, and Epidemic.

### Variables Excluded Initially
Demand, Inventory Level, and Units Ordered.

The final adjustment set will depend on variable definitions,
measurement timing, and causal assumptions.

## 4. Preliminary Findings: Pricing and Promotions

### Discount Analysis
- Higher discount levels were generally associated with higher average sales.
- Average selling prices tended to decrease as discounts increased.

### Promotion Analysis
- Records without promotion: 51,000
- Records with promotion: 25,000
- Average sales without promotion: 81.83 units
- Average sales with promotion: 103.10 units
- Average price without promotion: 68.92
- Average price with promotion: 65.30

### Competitor Pricing
- Correlation between Price and Competitor Pricing: 0.977.
- This indicates a very strong positive linear association.

### Important Limitation
These findings are descriptive. The timing and causal role of potential control variables must be established before DML estimation.

In [20]:
# Inspect data types of proposed DML variables

candidate_columns = [
    "Price",
    "Units Sold",
    "Store ID",
    "Product ID",
    "Date",
    "Promotion",
    "Competitor Pricing",
    "Weather Condition",
    "Seasonality",
    "Region",
    "Discount",
    "Category",
    "Epidemic"
]

variable_summary = pd.DataFrame({
    "Column": candidate_columns,
    "Data Type": [df[col].dtype for col in candidate_columns],
    "Unique Values": [df[col].nunique() for col in candidate_columns]
})

display(variable_summary)

,Column,Data Type,Unique Values
0,Price,float64,15396
1,Units Sold,int64,330
2,Store ID,str,5
3,Product ID,str,20
4,Date,datetime64[us],760
5,Promotion,int64,2
6,Competitor Pricing,float64,15963
7,Weather Condition,str,4
8,Seasonality,str,4
9,Region,str,4


In [21]:
# Examine dataset time coverage

print("First date:", df["Date"].min())
print("Last date:", df["Date"].max())

print("Number of unique dates:", df["Date"].nunique())

print("\nRecords per date:")
display(
    df.groupby("Date").size().describe().round(2)
)

First date: 2022-01-01 00:00:00
Last date: 2024-01-30 00:00:00
Number of unique dates: 760

Records per date:


count    760.0
mean     100.0
std        0.0
min      100.0
25%      100.0
50%      100.0
75%      100.0
max      100.0
dtype: float64

In [22]:
print("First date:", df["Date"].min())
print("Last date:", df["Date"].max())
print("Unique dates:", df["Date"].nunique())
print("Records per date:", df.groupby("Date").size().mean())

First date: 2022-01-01 00:00:00
Last date: 2024-01-30 00:00:00
Unique dates: 760
Records per date: 100.0
